# Aula 1 — Fundamentos de Big Data e Apache Spark

**Disciplina:** Big Data Processing — MBA Engenharia de Dados (Mackenzie)

**Objetivo:** Explorar dados de vendas da DataFlow Analytics usando PySpark.

---

## Instruções

1. Execute cada célula sequencialmente (Shift+Enter)
2. Leia os comentários e as explicações em Markdown
3. Ao final, resolva o **Desafio** proposto

---

## 1. Configuração — Criar a SparkSession

A SparkSession é o ponto de entrada para toda interação com o Spark.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFlow-Aula01") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"SparkSession criada com sucesso!")
print(f"  Versao: {spark.version}")
print(f"  App: {spark.sparkContext.appName}")
print(f"  Master: {spark.sparkContext.master}")

## 2. Carga de Dados

Vamos carregar o arquivo `vendas_2023.csv` com ~100K registros de vendas da ShopBrasil.

In [ ]:
# Ler o CSV de vendas
df_vendas = spark.read.csv(
    "/home/jovyan/work/data/aula_01/vendas_2023.csv",
    header=True,
    inferSchema=True
)

print(f"Registros carregados: {df_vendas.count():,}")
print(f"Colunas: {len(df_vendas.columns)}")

## 3. Exploração do Schema

Antes de analisar, precisamos entender a estrutura dos dados.

In [ ]:
# Estrutura do DataFrame
df_vendas.printSchema()

In [ ]:
# Primeiros registros
df_vendas.show(5, truncate=False)

In [ ]:
# Estatisticas descritivas das colunas numericas
df_vendas.describe().show()

## 4. Selecionar Colunas

Nem sempre precisamos de todas as colunas. `select()` projeta apenas o necessário.

In [ ]:
# Selecionar apenas colunas de interesse
df_resumo = df_vendas.select("order_id", "total_amount", "shipping_state", "payment_method")
df_resumo.show(5)

## 5. Filtros

Usar `filter()` ou `where()` para selecionar subconjuntos de dados.

In [ ]:
from pyspark.sql.functions import col

# Filtrar pedidos de Sao Paulo
df_sp = df_vendas.filter(col("shipping_state") == "SP")
print(f"Pedidos em SP: {df_sp.count():,}")

# Filtrar pedidos acima de R$ 500
df_alto_valor = df_vendas.filter(col("total_amount") > 500)
print(f"Pedidos > R$500: {df_alto_valor.count():,}")

# Combinar filtros (AND)
df_sp_alto = df_vendas.filter(
    (col("shipping_state") == "SP") & (col("total_amount") > 500)
)
print(f"Pedidos SP > R$500: {df_sp_alto.count():,}")

## 6. Agregações com groupBy

Agora vamos gerar os relatórios que a DataFlow precisa: faturamento por estado, por método de pagamento, etc.

In [ ]:
from pyspark.sql.functions import sum, avg, count, round, desc

# 6.1 Faturamento por estado
faturamento_estado = df_vendas \
    .groupBy("shipping_state") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento_total"),
        count("order_id").alias("total_pedidos"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento_total"))

print("=== FATURAMENTO POR ESTADO ===")
faturamento_estado.show(10)

In [ ]:
# 6.2 Vendas por metodo de pagamento
vendas_pagamento = df_vendas \
    .groupBy("payment_method") \
    .agg(
        count("order_id").alias("total_pedidos"),
        round(sum("total_amount"), 2).alias("faturamento"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("total_pedidos"))

print("=== VENDAS POR METODO DE PAGAMENTO ===")
vendas_pagamento.show()

In [ ]:
# 6.3 Vendas por status do pedido
vendas_status = df_vendas \
    .groupBy("status") \
    .agg(
        count("order_id").alias("total_pedidos"),
        round(sum("total_amount"), 2).alias("faturamento"),
    ) \
    .orderBy(desc("total_pedidos"))

print("=== PEDIDOS POR STATUS ===")
vendas_status.show()

## 7. Pipeline Encadeado

O estilo profissional no Spark é encadear operações num pipeline fluente.

In [ ]:
# Pipeline completo: Top 5 cidades de SP por faturamento
top_cidades_sp = df_vendas \
    .filter(col("shipping_state") == "SP") \
    .groupBy("shipping_city") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento"),
        count("order_id").alias("pedidos")
    ) \
    .orderBy(desc("faturamento")) \
    .limit(5)

print("=== TOP 5 CIDADES DE SP ===")
top_cidades_sp.show()

In [ ]:
# Segmentacao de clientes por faixa de valor
from pyspark.sql.functions import when

df_segmentado = df_vendas \
    .withColumn("segmento",
        when(col("total_amount") < 50, "Baixo")
        .when(col("total_amount") < 200, "Medio")
        .otherwise("Alto")
    )

segmentacao = df_segmentado \
    .groupBy("segmento") \
    .agg(
        count("*").alias("qtd_pedidos"),
        round(sum("total_amount"), 2).alias("faturamento"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento"))

print("=== SEGMENTACAO POR VALOR ===")
segmentacao.show()

## 8. Encerramento

Sempre fechar a SparkSession ao finalizar.

In [ ]:
spark.stop()
print("SparkSession encerrada.")

---

# DESAFIO

## Analise de Performance: pandas vs Spark

Agora e com voce! Implemente o seguinte:

### Requisitos:

1. **Carregue o mesmo arquivo** `vendas_2023.csv` usando **pandas** e usando **PySpark**
2. **Faca a mesma analise** em ambos: faturamento por estado (groupBy + sum + count + avg)
3. **Meça o tempo** de cada abordagem usando `time.time()`
4. **Compare os resultados** numa tabela
5. **Responda**: Para qual volume de dados o Spark comeca a valer a pena?

### Estrutura sugerida:

```python
import time
import pandas as pd

# --- PANDAS ---
start = time.time()
# ... seu codigo pandas aqui ...
tempo_pandas = time.time() - start

# --- SPARK ---
start = time.time()
# ... seu codigo spark aqui ...
tempo_spark = time.time() - start

print(f"Pandas: {tempo_pandas:.2f}s")
print(f"Spark:  {tempo_spark:.2f}s")
```

### Bonus:
- Tente com subconjuntos de dados (10K, 50K, 100K registros) e plote um grafico
- Use `%%time` magic do Jupyter para medir tempo de celula

---

**Boa sorte!** Este exercicio ajuda a entender quando migrar de pandas para Spark.

In [ ]:
# ===========================================================
# SEU CODIGO DO DESAFIO AQUI
# ===========================================================

